In [31]:
import requests
from lakehouse.daft import bronze
import daft

In [32]:
CATALOG = "daft_catalog"

# 1. Set Up

In [33]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [34]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"

bronze_instance = StarWarsBronze(**options)

In [35]:
(
    bronze_instance.load()
    .transform()
    .write(mode="overwrite")
    .execute("people", "planets")
)

2025-03-15 22:15:17 | people | execute | Started
2025-03-15 22:15:17 | people | load | Started
2025-03-15 22:15:22 | people | load | Completed in 0.07 min
2025-03-15 22:15:22 | people | transform | Started
2025-03-15 22:15:22 | people | transform | Completed in 0.0 min
2025-03-15 22:15:22 | people | write | Started
2025-03-15 22:15:22 | people | write | Completed in 0.0 min
2025-03-15 22:15:22 | people | execute | Completed in 0.07 min
2025-03-15 22:15:22 | planets | execute | Started
2025-03-15 22:15:22 | planets | load | Started
2025-03-15 22:15:25 | planets | load | Completed in 0.05 min
2025-03-15 22:15:25 | planets | transform | Started
2025-03-15 22:15:25 | planets | transform | Completed in 0.0 min
2025-03-15 22:15:25 | planets | write | Started
2025-03-15 22:15:25 | planets | write | Completed in 0.0 min
2025-03-15 22:15:25 | planets | execute | Completed in 0.05 min


In [36]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-15 22:15:22.523912,Luke Skywalker,1,https://www.swapi.tech/api/people/1
2025-03-15 22:15:22.523912,C-3PO,2,https://www.swapi.tech/api/people/2
2025-03-15 22:15:22.523912,R2-D2,3,https://www.swapi.tech/api/people/3
2025-03-15 22:15:22.523912,Darth Vader,4,https://www.swapi.tech/api/people/4
2025-03-15 22:15:22.523912,Leia Organa,5,https://www.swapi.tech/api/people/5
2025-03-15 22:15:22.523912,Owen Lars,6,https://www.swapi.tech/api/people/6
2025-03-15 22:15:22.523912,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7
2025-03-15 22:15:22.523912,R5-D4,8,https://www.swapi.tech/api/people/8


No. Rows: 82


In [37]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-15 22:15:25.828508,Tatooine,1,https://www.swapi.tech/api/planets/1
2025-03-15 22:15:25.828508,Alderaan,2,https://www.swapi.tech/api/planets/2
2025-03-15 22:15:25.828508,Yavin IV,3,https://www.swapi.tech/api/planets/3
2025-03-15 22:15:25.828508,Hoth,4,https://www.swapi.tech/api/planets/4
2025-03-15 22:15:25.828508,Dagobah,5,https://www.swapi.tech/api/planets/5
2025-03-15 22:15:25.828508,Bespin,6,https://www.swapi.tech/api/planets/6
2025-03-15 22:15:25.828508,Endor,7,https://www.swapi.tech/api/planets/7
2025-03-15 22:15:25.828508,Naboo,8,https://www.swapi.tech/api/planets/8


No. Rows: 60


In [38]:
bronze_instance.data["people"].show()

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-15 22:15:22.523912,Luke Skywalker,1,https://www.swapi.tech/api/people/1
2025-03-15 22:15:22.523912,C-3PO,2,https://www.swapi.tech/api/people/2
2025-03-15 22:15:22.523912,R2-D2,3,https://www.swapi.tech/api/people/3
2025-03-15 22:15:22.523912,Darth Vader,4,https://www.swapi.tech/api/people/4
2025-03-15 22:15:22.523912,Leia Organa,5,https://www.swapi.tech/api/people/5
2025-03-15 22:15:22.523912,Owen Lars,6,https://www.swapi.tech/api/people/6
2025-03-15 22:15:22.523912,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7
2025-03-15 22:15:22.523912,R5-D4,8,https://www.swapi.tech/api/people/8


In [39]:
bronze_instance.data["planets"].show()

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-15 22:15:25.828508,Tatooine,1,https://www.swapi.tech/api/planets/1
2025-03-15 22:15:25.828508,Alderaan,2,https://www.swapi.tech/api/planets/2
2025-03-15 22:15:25.828508,Yavin IV,3,https://www.swapi.tech/api/planets/3
2025-03-15 22:15:25.828508,Hoth,4,https://www.swapi.tech/api/planets/4
2025-03-15 22:15:25.828508,Dagobah,5,https://www.swapi.tech/api/planets/5
2025-03-15 22:15:25.828508,Bespin,6,https://www.swapi.tech/api/planets/6
2025-03-15 22:15:25.828508,Endor,7,https://www.swapi.tech/api/planets/7
2025-03-15 22:15:25.828508,Naboo,8,https://www.swapi.tech/api/planets/8


# 6 Clean Up

In [40]:
import shutil
shutil.rmtree(f"D:/Data/{CATALOG}")